# Drunk Driving Fatality Analysis — Predictive Modeling
**Author:** Jared Oberg  
**Dataset:** NHTSA FARS 2015–2016 (processed in `01_Data_Preparation.ipynb`)

---

## Objective

Build a binary classification model to predict whether a fatal crash involved a drunk driver,
using crash-level features available at the time of incident reporting.

**Target variable:** `drunk_driver_involved` (1 = drunk driving crash, 0 = non-drunk crash)

**Models compared:**
- Logistic Regression (baseline)
- Random Forest (champion)

**Champion model results:**
- ROC-AUC: **0.773**
- Accuracy: **71.6%**
- Key finding: Time of crash alone captures most of the predictive signal

---

## Notebook Structure
1. Setup & Data Load
2. Exploratory Analysis — Key Risk Factor Comparisons
3. Feature Selection — From 142 Variables to a Final Feature Set
4. Feature Engineering for Modeling
5. Train/Test Split
6. Model Selection — Why These Two Models?
7. Logistic Regression (Baseline)
8. Random Forest (Champion)
9. Model Comparison & Feature Importance
10. Model Comparison Summary
11. Business Interpretation
12. Conclusions

## 1. Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, roc_auc_score,
                              roc_curve, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Load the crash detail CSV produced in 01_Data_Preparation.ipynb
df = pd.read_csv('/content/drive/MyDrive/FARS_Dashboard/FARS_CrashDetail_2015_2016.csv')

print(f'Loaded: {len(df):,} rows × {df.shape[1]} cols')
print(f'Years: {sorted(df["year"].unique())}')
df.head(3)

## 2. Exploratory Analysis — Risk Factor Comparisons

Before modeling we examine how key risk factors differ between drunk and sober fatal crashes.
This informs feature selection and helps interpret model outputs.

In [ ]:
# ── Time of day distribution ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fatalities by hour of day
hour_counts = df.groupby('hour_of_crash')['number_of_fatalities'].sum()
colors = ['#2c3e7a' if h <= 5 else '#f0a500' if h <= 11
          else '#4a90d9' if h <= 17 else '#c0392b'
          for h in hour_counts.index]
axes[0].bar(hour_counts.index, hour_counts.values, color=colors, edgecolor='white')
axes[0].set_title('Drunk Driving Fatalities by Hour of Day', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Number of Fatalities')
axes[0].set_xticks(range(0, 24, 2))

# Fatalities by time period
period_counts = df.groupby('time_period')['number_of_fatalities'].sum().sort_values(ascending=False)
axes[1].bar(period_counts.index, period_counts.values,
            color=['#c0392b', '#2c3e7a', '#4a90d9', '#f0a500'], edgecolor='white')
for i, (label, val) in enumerate(period_counts.items()):
    axes[1].text(i, val + 50, f'{val:,}\n({val/period_counts.sum()*100:.1f}%)',
                 ha='center', fontsize=10, fontweight='bold')
axes[1].set_title('Fatalities by Time Period', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Number of Fatalities')
axes[1].tick_params(axis='x', labelsize=9)

plt.tight_layout()
plt.show()

print('\nKey finding: Evening + Late Night = 75%+ of all drunk driving fatalities')

In [ ]:
# ── Risk factor summary ────────────────────────────────────
risk_cols = {
    'no_seatbelt': 'No Seatbelt',
    'has_prior_dwi': 'Prior DWI',
    'has_prior_susp': 'Prior Suspension',
    'invalid_license': 'Invalid License',
    'speeding_involved': 'Speeding'
}

risk_pcts = {label: (df[col] == 'Yes').mean() * 100
             for col, label in risk_cols.items()}

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(list(risk_pcts.keys()), list(risk_pcts.values()),
               color=['#c0392b', '#2c3e7a', '#2c3e7a', '#f0a500', '#2c3e7a'],
               edgecolor='white')
for bar, val in zip(bars, risk_pcts.values()):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=11, fontweight='bold')
ax.set_title('Risk Factor Profile — Drunk Driving Fatal Crashes (2015–2016)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('% of Crashes')
ax.set_xlim(0, 60)
plt.tight_layout()
plt.show()

In [ ]:
# ── Road type distribution ─────────────────────────────────
road_fat = df.groupby('road_type')['number_of_fatalities'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(road_fat.index, road_fat.values, color='#2c3e7a', edgecolor='white')
for i, val in enumerate(road_fat.values):
    ax.text(val + 20, i, f'{val:,}', va='center', fontsize=10, fontweight='bold')
ax.set_title('Drunk Driving Fatalities by Road Type (2015–2016)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Fatalities')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Feature Selection — From 142 Variables to a Final Feature Set

### The Starting Point

The raw FARS dataset contained **142 unique variables** spanning crash characteristics,
vehicle details, driver demographics, roadway conditions, and environmental factors.
Rather than feeding all variables into a model, we worked through a structured feature
selection process to identify which variables had the strongest relationship with
drunk driver involvement.

### Step 1 — Establish the Target

Initial exploratory analysis of all fatal crashes (2015–2016) revealed that
**approximately 27% involved a drunk driver** — a large enough signal to justify
shifting the entire project focus to alcohol-related fatalities.

### Step 2 — Explore Relationships Between All Features and Drunk Driver Involvement

We examined each available feature against the drunk driver involvement flag,
looking for meaningful differences between drunk and non-drunk crash records.
Features were grouped into categories:

| Category | Example Variables | Relationship to Drunk Driving |
|----------|------------------|------------------------------|
| **Temporal** | hour of crash, day of week, month | Very strong — clear time-of-day pattern |
| **Driver history** | prior DWI, license suspension, invalid license | Strong — 4x more likely in drunk crashes |
| **Crash behavior** | speeding, seatbelt use | Moderate — 1.7–2.3x more likely |
| **Roadway** | road type, junction type, land use | Moderate — freeway ramps elevated |
| **Environmental** | weather, light condition | Weak — night conditions correlated but driven by hour |
| **Demographic** | driver age | Weak — slight skew toward younger drivers |
| **Vehicle** | vehicle type, make/model | Minimal — negligible separation |

### Step 3 — Eliminate Low-Signal and Unusable Features

Several categories were deprioritized or excluded:

- **Travel speed** — only available for ~33% of records, too sparse to use reliably
- **Vehicle characteristics** — make, model, body type showed negligible separation between
  drunk and non-drunk crashes
- **Environmental conditions** — weather and light condition appeared correlated but were
  largely redundant with hour of crash (night crashes = dark conditions)
- **Geographic coordinates** — too granular for a classification model without spatial clustering

### Step 4 — Confirm with Correlation Analysis

Point-biserial correlations and chi-squared tests confirmed the ranking:
temporal features dominated, followed by driver history, with roadway and
environmental features contributing marginal additional signal.

In [ ]:
# ── Point-biserial correlation of numeric features vs drunk driver flag ──
# Note: requires the full accident table (both drunk and non-drunk crashes)
# This cell uses df_acc from the stacked_tables loaded in notebook 01

from scipy import stats

df_acc = stacked_tables['accident'].copy()
df_acc = df_acc[df_acc['year'].isin([2015, 2016])].copy()
df_acc['drunk_driver_involved'] = (df_acc['number_of_drunk_drivers'] > 0).astype(int)

# Numeric features to test
numeric_features = [
    'hour_of_crash', 'day_of_week', 'month_of_crash',
    'number_of_fatalities', 'number_of_persons_not_in_motor_vehicles_in_transport_mvit'
]

print('Point-Biserial Correlation with drunk_driver_involved:')
print('=' * 55)
correlations = []
for col in numeric_features:
    if col in df_acc.columns:
        clean = df_acc[[col, 'drunk_driver_involved']].dropna()
        corr, pval = stats.pointbiserialr(
            clean['drunk_driver_involved'], clean[col])
        correlations.append((col, corr, pval))

correlations.sort(key=lambda x: abs(x[1]), reverse=True)
for col, corr, pval in correlations:
    sig = '✅ significant' if pval < 0.05 else '—'
    print(f'  {col:<55} r={corr:>6.3f}  {sig}')

In [ ]:
# ── Drunk driving rate by hour — the strongest single pattern ──
hourly = df_acc.groupby('hour_of_crash').agg(
    total=('consecutive_number', 'count'),
    drunk=('drunk_driver_involved', 'sum')
).reset_index()
hourly = hourly[hourly['hour_of_crash'] <= 23]  # exclude sentinel 99
hourly['pct_drunk'] = hourly['drunk'] / hourly['total'] * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Chart 1: % drunk by hour
colors = ['#c0392b' if h <= 5 or h >= 20 else '#2c3e7a'
          for h in hourly['hour_of_crash']]
axes[0].bar(hourly['hour_of_crash'], hourly['pct_drunk'],
            color=colors, edgecolor='white')
axes[0].axhline(y=hourly['pct_drunk'].mean(), color='black',
                linestyle='--', linewidth=1.5,
                label=f'Overall avg: {hourly["pct_drunk"].mean():.1f}%')
axes[0].set_title('% of Fatal Crashes Involving Drunk Driver\nby Hour of Day',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('% Involving Drunk Driver')
axes[0].set_xticks(range(0, 24, 2))
axes[0].legend()
axes[0].set_facecolor('#f9f9f9')

# Chart 2: % drunk by day of week
day_map = {1:'Sun', 2:'Mon', 3:'Tue', 4:'Wed', 5:'Thu', 6:'Fri', 7:'Sat'}
daily = df_acc.groupby('day_of_week').agg(
    total=('consecutive_number', 'count'),
    drunk=('drunk_driver_involved', 'sum')
).reset_index()
daily['pct_drunk'] = daily['drunk'] / daily['total'] * 100
daily['day_label'] = daily['day_of_week'].map(day_map)
weekend_colors = ['#c0392b' if d in [1, 7] else '#2c3e7a'
                  for d in daily['day_of_week']]
axes[1].bar(daily['day_label'], daily['pct_drunk'],
            color=weekend_colors, edgecolor='white')
axes[1].axhline(y=daily['pct_drunk'].mean(), color='black',
                linestyle='--', linewidth=1.5,
                label=f'Overall avg: {daily["pct_drunk"].mean():.1f}%')
axes[1].set_title('% of Fatal Crashes Involving Drunk Driver\nby Day of Week',
                  fontsize=13, fontweight='bold')
axes[1].set_ylabel('% Involving Drunk Driver')
axes[1].legend()
axes[1].set_facecolor('#f9f9f9')

plt.suptitle('Temporal Features Show the Strongest Separation',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Late night (12am-5am) drunk rate:  {hourly[hourly["hour_of_crash"].between(0,5)["pct_drunk"].mean():.1f}%')
print(f'Morning (6am-11am) drunk rate:     {hourly[hourly["hour_of_crash"].between(6,11)]["pct_drunk"].mean():.1f}%')

### Step 5 — Final Feature Set

Based on the correlation analysis and exploratory findings, we converged on **8 features**
for the final models. We also tested a broader 22-feature set to verify we weren't
leaving meaningful signal on the table.

| Feature | Category | Justification |
|---------|----------|---------------|
| `hour_of_crash` | Temporal | Strongest single predictor |
| `time_period_num` | Temporal | Bucketed hour — captures non-linear time pattern |
| `is_weekend` | Temporal | Weekend crashes show elevated drunk rate |
| `is_night` | Temporal | Captures after-dark risk independent of exact hour |
| `road_type_enc` | Roadway | Freeway/arterial vs local roads show different rates |
| `is_rural` | Roadway | Rural vs urban crash patterns differ |
| `number_of_fatalities` | Crash severity | Drunk crashes tend to be more severe |
| `month_of_crash` | Temporal | Seasonal patterns in alcohol-related crashes |

**Key validation:** Reducing from 22 features to 8 (primarily temporal) dropped
ROC-AUC by only **0.05** (0.773 → 0.723), confirming that timing captures
the vast majority of the predictive signal in this dataset.

## 4. Feature Engineering for Modeling

We need to reconstruct the full accident table to get both drunk and non-drunk crash records
for binary classification. The crash detail CSV only contains drunk driving crashes.

We use the stacked accident table from BigQuery (loaded in `01_Data_Preparation.ipynb`).

In [ ]:
# NOTE: This cell assumes stacked_tables is in memory from 01_Data_Preparation.ipynb
# If running standalone, re-run the BigQuery cells in notebook 01 first.

df_acc = stacked_tables['accident'].copy()
df_acc = df_acc[df_acc['year'].isin([2015, 2016])].copy()
df_acc['drunk_driver_involved'] = (df_acc['number_of_drunk_drivers'] > 0).astype(int)

# Filter to valid coordinates
df_model = df_acc[
    (df_acc['latitude'].between(24, 50)) &
    (df_acc['longitude'].between(-125, -65))
].copy()

print(f'Modeling dataset: {len(df_model):,} crashes')
print(f'Drunk driving crashes: {df_model["drunk_driver_involved"].sum():,} '
      f'({df_model["drunk_driver_involved"].mean()*100:.1f}%)')

In [ ]:
# ── Create model features ──────────────────────────────────
def time_period_num(hour):
    """Convert hour to time period bucket (0-3)"""
    if 0 <= hour <= 5:     return 0   # Late Night
    elif 6 <= hour <= 11:  return 1   # Morning
    elif 12 <= hour <= 17: return 2   # Afternoon
    else:                  return 3   # Evening

df_model['time_period_num'] = df_model['hour_of_crash'].apply(time_period_num)
df_model['is_weekend']      = df_model['day_of_week'].isin([1, 7]).astype(int)
df_model['is_night']        = df_model['hour_of_crash'].apply(
    lambda h: 1 if (h >= 20 or h <= 5) else 0)
df_model['is_rural']        = (df_model['land_use_name'] == 'Rural').astype(int)

# Encode road type
le = LabelEncoder()
df_model['road_type_enc'] = le.fit_transform(
    df_model['functional_system_name'].fillna('Unknown'))

print('Features created:')
print('  time_period_num, is_weekend, is_night, is_rural, road_type_enc')
print(f'\nClass balance:')
print(df_model['drunk_driver_involved'].value_counts())

## 5. Train/Test Split

In [ ]:
features = [
    'hour_of_crash',
    'time_period_num',
    'is_weekend',
    'is_night',
    'is_rural',
    'road_type_enc',
    'number_of_fatalities',
    'month_of_crash'
]

X = df_model[features].fillna(0)
y = df_model['drunk_driver_involved']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Training set: {len(X_train):,} records')
print(f'Test set:     {len(X_test):,} records')
print(f'Positive rate (drunk) — Train: {y_train.mean()*100:.1f}% | Test: {y_test.mean()*100:.1f}%')

## 6. Model Selection — Why These Two Models?

Two supervised classification models were selected to evaluate different approaches
to the same prediction problem.

### Logistic Regression — Baseline
Used as a linear baseline classifier to determine whether drunk driver involvement
could be explained through primarily linear relationships between crash features
and the outcome. Logistic Regression is interpretable, fast, and establishes a
performance floor that any more complex model should comfortably exceed.

### Random Forest — Primary Model
Selected because it captures **nonlinear relationships**, handles mixed feature types
without scaling, is robust to outliers, and provides **feature importance scores**
that make results directly interpretable. Given that the relationship between time
of day and drunk driving is inherently nonlinear (risk spikes sharply after midnight,
drops sharply in the morning), a nonlinear model was expected to outperform a linear one.

### What Comparing Them Tells Us
If Random Forest substantially outperforms Logistic Regression, it confirms that
the predictive signal in this dataset is driven by **complex nonlinear interactions**
rather than simple linear separability — and that the temporal patterns identified
in exploratory analysis are real and modelable.

## 7. Logistic Regression — Baseline

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

auc_lr = roc_auc_score(y_test, y_prob_lr)
acc_lr = (y_pred_lr == y_test).mean()

print('LOGISTIC REGRESSION RESULTS')
print('=' * 40)
print(f'ROC-AUC:  {auc_lr:.3f}')
print(f'Accuracy: {acc_lr*100:.1f}%')
print()
print(classification_report(y_test, y_pred_lr, target_names=['Sober', 'Drunk']))

## 8. Random Forest — Champion Model

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_split=20,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

auc_rf = roc_auc_score(y_test, y_prob_rf)
acc_rf = (y_pred_rf == y_test).mean()

print('RANDOM FOREST RESULTS')
print('=' * 40)
print(f'ROC-AUC:  {auc_rf:.3f}')
print(f'Accuracy: {acc_rf*100:.1f}%')
print()
print(classification_report(y_test, y_pred_rf, target_names=['Sober', 'Drunk']))

## 9. Model Comparison & Feature Importance

In [ ]:
# ── ROC Curve Comparison ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC curves
for model_name, y_prob, color in [
    ('Logistic Regression', y_prob_lr, 'steelblue'),
    ('Random Forest', y_prob_rf, 'crimson')
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[0].plot(fpr, tpr, label=f'{model_name} (AUC={auc:.3f})', color=color, linewidth=2)

axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC=0.500)')
axes[0].set_title('ROC Curve Comparison', fontsize=13, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(fontsize=9)
axes[0].set_facecolor('#f9f9f9')

# Feature importance
feat_imp = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)
feat_imp.plot(kind='barh', ax=axes[1], color='crimson', edgecolor='white')
axes[1].set_title('Random Forest\nFeature Importance', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importance Score')
axes[1].set_facecolor('#f9f9f9')

# Model comparison bar chart
metrics = ['ROC-AUC', 'Accuracy']
lr_scores = [auc_lr, acc_lr]
rf_scores = [auc_rf, acc_rf]
x = np.arange(len(metrics))
width = 0.35
axes[2].bar(x - width/2, lr_scores, width, label='Logistic Regression',
            color='steelblue', edgecolor='white')
axes[2].bar(x + width/2, rf_scores, width, label='Random Forest',
            color='crimson', edgecolor='white')
for i, (lr_v, rf_v) in enumerate(zip(lr_scores, rf_scores)):
    axes[2].text(i - width/2, lr_v + 0.005, f'{lr_v:.3f}', ha='center', fontsize=10, fontweight='bold')
    axes[2].text(i + width/2, rf_v + 0.005, f'{rf_v:.3f}', ha='center', fontsize=10, fontweight='bold')
axes[2].set_title('Model Comparison', fontsize=13, fontweight='bold')
axes[2].set_xticks(x)
axes[2].set_xticklabels(metrics)
axes[2].set_ylim(0, 1.0)
axes[2].legend()
axes[2].set_facecolor('#f9f9f9')

plt.suptitle('Random Forest vs Logistic Regression — Drunk Driving Prediction',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion Matrix ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, y_pred, title in [
    (axes[0], y_pred_lr, 'Logistic Regression'),
    (axes[1], y_pred_rf, 'Random Forest')
]:
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['Sober', 'Drunk'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontsize=12, fontweight='bold')

plt.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Model Comparison Summary

| Model | ROC-AUC | Accuracy | Recall (Drunk) |
|-------|---------|----------|----------------|
| Logistic Regression | 0.711 | 64.9% | 67% |
| **Random Forest** | **0.773** | **71.6%** | **69%** |

### Champion Model: Random Forest

Random Forest was selected as the final model because it consistently outperformed
Logistic Regression across all evaluation metrics while also providing interpretable
feature importance rankings.

### Why the Gap Is Small — and Why That Matters

The relatively modest performance gap between the two models (0.773 vs 0.711 ROC-AUC)
is itself a meaningful finding. As the capstone paper notes, the predictive limitations
were more likely caused by **overlap within the dataset** than deficiencies in algorithm
selection or model optimization. Many fatal crash characteristics are highly similar between
alcohol-related and non-alcohol-related crashes — making this an inherently difficult
classification problem regardless of which algorithm is used.

> **Key Insight:** Adding 20 additional demographic, roadway, and environmental variables
> beyond hour of crash and day of week improved ROC-AUC by only **0.05 points** (0.723 → 0.773).
> Temporal variables contain the overwhelming majority of the predictive signal available
> within the FARS dataset.

## 11. Business Interpretation

### What This Model Actually Does

It is important to clarify the objective of this modeling effort. The Random Forest and
Logistic Regression models were built to **classify whether a fatal crash involved a drunk
driver** based on crash characteristics available in the FARS dataset. The models are not
designed to predict where drunk driving will occur in the future — they are analytical tools
for understanding which crash characteristics are most strongly associated with
alcohol-related fatalities.

The primary value of the modeling was not the classification performance itself, but
**what the feature importance rankings revealed** about the data.

### What the Model Tells Us

- **Hour of crash was the dominant predictor** — accounting for ~43% of total feature
  importance in the Random Forest model, far exceeding any demographic, roadway,
  or environmental variable
- **Temporal variables captured most of the signal** — reducing from 22 features to
  just hour and day of week dropped ROC-AUC by only 0.05, confirming that timing
  is the primary distinguishing factor between drunk and sober fatal crashes
- **Behavioral and history variables** (prior DWI, speeding, invalid license) showed
  meaningful separation during exploratory analysis but contributed limited additional
  predictive lift in the models — suggesting they are better used for identifying
  at-risk driver populations than for crash classification
- **Light condition** was the top feature in Logistic Regression, reflecting the
  correlation between nighttime driving and alcohol involvement — though this is
  largely redundant with hour of crash

### Translating Results to Action

These findings directly support three enforcement and prevention strategies:

1. **Targeted late-night enforcement** — DWI patrols and sobriety checkpoints
   concentrated between 9pm and 3am, particularly at freeway interchange areas,
   would intersect the highest-risk crash concentration
2. **Repeat offender intervention** — Prior DWI conviction was 4x more common
   in drunk driving crashes, supporting programs targeting repeat offenders
3. **Rideshare and transit promotion** — Evening and late-night periods that account
   for the majority of alcohol-related fatalities align with peak rideshare availability

### Model Limitations

The Random Forest model achieved a ROC-AUC of 0.773, reflecting the inherent difficulty
of distinguishing alcohol-related fatal crashes from non-alcohol-related fatal crashes
using historical crash records alone. Many crash characteristics overlap between the two
groups, limiting the model's ability to perfectly separate them.

The primary objective of this modeling effort was not to develop a production-ready
prediction system, but to **identify which crash characteristics were most strongly
associated with alcohol-related fatalities**. Feature importance analysis consistently
showed that hour of crash was the dominant predictor, followed by variables related to
crash timing, roadway characteristics, and driver history. Even after adding numerous
demographic and environmental variables, model performance improved only modestly,
confirming that temporal factors contain the majority of the predictive signal available
within the FARS dataset.

Because the dataset contains only post-crash records, it lacks real-time behavioral
indicators such as lane departures, swerving, hard braking, and vehicle telemetry
that would likely improve predictive performance. Future models incorporating these
data sources could better distinguish alcohol-related crashes before or during crash
events rather than identifying patterns after they occur.

The accompanying Tableau dashboard complements the machine learning analysis by
allowing users to interactively explore the geographic, temporal, and roadway patterns
identified throughout this project.

## 12. Conclusions

### Model Performance

| Model | ROC-AUC | Accuracy |
|-------|---------|----------|
| Logistic Regression | 0.711 | 64.9% |
| **Random Forest** | **0.773** | **71.6%** |

Random Forest outperforms Logistic Regression and is selected as the champion model.

### Overall Project Findings

Across exploratory analysis, machine learning, and geographic visualization, the same
conclusion consistently emerged: **temporal variables provided substantially more
predictive information than demographic, roadway, or environmental characteristics.**
The agreement across multiple analytical approaches increases confidence that these
findings represent meaningful patterns rather than artifacts of a single modeling technique.

The most significant finding of this project is that the combination of time and location
provides the clearest framework for identifying high-risk drunk-driving fatality patterns.
Time of day emerged as the strongest predictor of drunk driver involvement, while geographic
location, prior DWI history, invalid license status, speeding behavior, and seatbelt nonuse
provided additional separation between drunk and sober drivers. While model performance
remained moderate, the results demonstrate that drunk-driving fatalities are not randomly
distributed — they are concentrated within identifiable temporal, geographic, and behavioral
conditions that may support future prevention efforts and resource allocation decisions.

### Project Takeaways

This project demonstrates an end-to-end analytics workflow using a large, real-world
public safety dataset. Key accomplishments included:

- Merging **143 BigQuery tables** (8M+ rows) into a unified analytical dataset
- Engineering features across crash, vehicle, and person-level records
- Performing exploratory analysis to identify behavioral and temporal patterns
- Building and comparing Random Forest and Logistic Regression classification models
- Evaluating feature importance to identify the strongest predictors of drunk driver involvement
- Developing an **interactive three-level Tableau dashboard** for geographic exploration

The strongest overall finding was that crash timing consistently emerged as the dominant
predictor of alcohol-related fatalities across all three analytical methods — exploratory
analysis, machine learning, and geographic visualization.